# 10.1 합성곱, 출력 크기, 파라미터 수와 FLOPs — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter10_1_conv_output_size.ipynb)

책 본문: [10.1 합성곱, 출력 크기, 파라미터 수와 FLOPs](https://smhanlab.com/book-ml/kor/ml1/chapter10/1.html)

이 노트북은 책 10.1절의 손계산을 코드로 재현합니다: (1) 3×3 필터가 5×5 이미지에서 수직선을 감지하는 `conv2d`를 직접 실행, (2) **출력 크기 공식** $n_{out} = \lfloor(n+2p-k)/s\rfloor + 1$을 "필터가 다 들어가는 위치를 세는 것"으로 검증, (3) 패딩/스트라이드 4가지 조합의 시각화, (4) **파라미터 수**와 **FLOPs**를 미니 LeNet 3층에서 층별로 계산하고 완전연결층과 비교, (5) FLOPs/파라미터 비율이 $n_{out}^2$임을 3×3 vs 1×1 필터 비교로 확인. numpy/matplotlib만 씁니다 — 모든 시드 고정.


In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)


## 1. 합성곱 한 번 돌려보기: 수직선 감지기

본문의 "손으로 한 번" 예(5×5 이미지, 2번 열에 수직 흰선, 중심 열만 1인 3×3 필터)를 그대로 실행합니다. 손계산으로 얻은 출력이 3×3이고 중심 열(출력의 j=1)이 3인 것을 코드가 확인합니다.


In [2]:
def conv2d(image, kernel):
    img_h, img_w = image.shape
    k = kernel.shape[0]
    out_h, out_w = img_h - k + 1, img_w - k + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = image[i:i+k, j:j+k].ravel() @ kernel.ravel()
    return output

# 5x5 이미지: 2번 열에 수직 흰선(밝기 1, 배경 0)
img = np.zeros((5, 5));  img[:, 2] = 1.0
# 3x3 "수직선 감지기": 중심 열만 1
kernel = np.zeros((3, 3));  kernel[:, 1] = 1.0

out = conv2d(img, kernel)
print("출력 크기:", out.shape)          # (3, 3)  <- n - k + 1 = 5 - 3 + 1
print("출력 지도:")
print(out.astype(int))
assert out.shape == (3, 3)
assert np.array_equal(out, [[0, 3, 0], [0, 3, 0], [0, 3, 0]])
print("\n손계산 표와 일치: 중심 위치 (1,1) = 3, 나머지 0 (중심 위치 하나만 직접 계산해본 것)")


출력 크기: (3, 3)
출력 지도:
[[0 3 0]
 [0 3 0]
 [0 3 0]]

손계산 표와 일치: 중심 위치 (1,1) = 3, 나머지 0 (중심 위치 하나만 직접 계산해본 것)


## 2. 출력 크기 공식: "필터가 다 들어가는 위치를 세자"

$$(n + 2p - k)$$  — 패딩을 두른 입력 길이(한 변당 p씩, 좌우 합쳐 **2p**)에서 필터 길이 k를 빼면, 필터 **왼쪽 끝**이 놓일 수 있는 범위가 $[0,\, n+2p-k]$다. 스트라이드 s로 걸러가며 세면 $\lfloor(n+2p-k)/s\rfloor + 1$개.


In [3]:
def out_size(n, k, p=0, s=1):
    return (n + 2*p - k) // s + 1       # 1D 한 방향

# (a) 8x8, k=3, p=0: 스트라이드만 바꿔가며 직접 세고 공식과 비교
for s, starts in [(1, [0,1,2,3,4,5]), (2, [0,2,4]), (4, [0,4])]:
    counted = len(starts)
    formula = out_size(8, 3, 0, s)
    print(f"s={s}: 직접 센 시작 위치 {starts} -> {counted}개   공식 -> {formula}개   일치={counted == formula}")
    assert counted == formula

# (b) 28x28, k=3: 패딩/스트라이드 4가지 조합 (본문 그림의 4개 panel)
print()
for p, s in [(0, 1), (1, 1), (0, 2), (1, 2)]:
    n_out = out_size(28, 3, p, s)
    print(f"n=28, k=3, p={p}, s={s}  ->  n_out = {n_out}")

# (c) 스트라이드 2가 실제로 하는 일: 224 -> 112 (AlexNet 첫 합성곱), 56 -> 28
print()
print("스트라이드 2: 224 ->", out_size(224, 3, 1, 2), "   56 ->", out_size(56, 3, 1, 2))


s=1: 직접 센 시작 위치 [0, 1, 2, 3, 4, 5] -> 6개   공식 -> 6개   일치=True
s=2: 직접 센 시작 위치 [0, 2, 4] -> 3개   공식 -> 3개   일치=True
s=4: 직접 센 시작 위치 [0, 4] -> 2개   공식 -> 2개   일치=True

n=28, k=3, p=0, s=1  ->  n_out = 26
n=28, k=3, p=1, s=1  ->  n_out = 28
n=28, k=3, p=0, s=2  ->  n_out = 13
n=28, k=3, p=1, s=2  ->  n_out = 14

스트라이드 2: 224 -> 112    56 -> 28


In [4]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"
n = 28; k = 3

def valid_starts(n, k, p, s):
    return list(range(0, n + 2*p - k + 1, s))

cases = [
    ("p=0, s=1", 0, 1),
    ("p=1, s=1 (same)", 1, 1),
    ("p=0, s=2", 0, 2),
    ("p=1, s=2", 1, 2),
]

fig, axes = plt.subplots(2, 2, figsize=(9, 5.2))
for ax, (title, p, s) in zip(axes.ravel(), cases):
    starts = valid_starts(n, k, p, s)
    cells = list(range(-p, n + p))
    for c in cells:
        col = "white" if 0 <= c < n else "#d0d0d0"
        ax.add_patch(plt.Rectangle((c, 0), 1, 1, facecolor=col, edgecolor="#888", lw=0.6))
    for st in starts:
        ax.add_patch(plt.Rectangle((st, 1.0), k, 0.35, facecolor="#fca5a5", edgecolor="#dc2626", lw=0.8, zorder=3))
        ax.plot([st, st], [1.0, 1.45], color="#dc2626", lw=1.0, zorder=4)
    ax.text(n/2 - p, -0.55, f"n_out = {len(starts)}", ha="center", fontsize=11, color="#1d4ed8", fontweight="bold")
    ax.set_xlim(-p - 0.3, n + p + 0.3)
    ax.set_ylim(-0.8, 1.9)
    ax.set_yticks([]); ax.set_xticks([])
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.set_title(f"{title}", fontsize=10.5)

fig.suptitle("28×28 입력, 3×3 필터: 유효 필터 시작 위치(빨간 틱) = 출력 크기 n_out", fontsize=12, y=1.00)
fig.text(0.5, -0.02, "흰 칸=원래 입력  |  회색 칸=패딩  |  빨간 막대/틱=필터가 완전히 들어가는 위치(= 출력 채널의 한 픽셀)",
         ha="center", fontsize=8.5, color="#444")
fig.tight_layout(rect=[0, 0.02, 1, 0.98])
fig.savefig(IMG + "/ch10_1_padding_strides.svg")
plt.show()


## 3. 파라미터 수와 FLOPs: 미니 LeNet 세 층을 층별로 계산

본문의 "손계산 표" 네트워크 — 28×28 입력, conv(3×3, $C_{out}$=4) → conv(3×3, 4→8) → 2×2 max pooling → conv(3×3, 8→16), 패딩 0, 스트라이드 1 — 을 층별로 $n_{out}$, 파라미터, FLOPs를 계산한다.

$$\text{params} = (k^2 C_{in} + 1)\, C_{out}, \qquad \text{FLOPs} = n_{out}^2\, C_{out}\, k^2 C_{in}$$


In [5]:
def conv_cost(n_in, k, c_in, c_out, p=0, s=1):
    n_out = out_size(n_in, k, p, s)
    params = (k*k*c_in + 1) * c_out        # bias 포함
    flops  = n_out**2 * c_out * (k*k*c_in) # 출력 위치마다 반복
    return n_out, params, flops

layers = [
    ("conv 1", 28, 3, 1, 4),
    ("conv 2", 26, 3, 4, 8),
    ("conv 3", 12, 3, 8, 16),
]
tot_p, tot_f = 0, 0
print(f"{'층':<8}{'n_in':>5}{'n_out':>7}{'params':>9}{'FLOPs':>10}")
for name, n, k, ci, co in layers:
    no, par, fl = conv_cost(n, k, ci, co)
    tot_p += par; tot_f += fl
    print(f"{name:<8}{n:>5}{no:>7}{par:>9,}{fl:>10,}")
print("-" * 42)
print(f"{'합계':<8}{'':>5}{'':>7}{tot_p:>9,}{tot_f:>10,}")

# 풀링: 파라미터 0, 비용은 비교 연산 (2x2 창 144*8개, 창당 3회 비교)
pool_cmp = (12*12*8) // 4 * 3
print(f"pool (2x2 max, s=2): 파라미터 0, 비교 연산 {pool_cmp}회 (FLOPs 합계에서 무시할 만큼 작음)")

# 대조: 같은 출력 체적(10x10x16 = 1600)을 완전연결층 하나로
mlp_params = 784 * 1600 + 1600
mlp_flops  = 784 * 1600
print()
print(f"완전연결 784 -> 1600: params {mlp_params:,}  FLOPs {mlp_flops:,}")
print(f"  params {mlp_params/tot_p:.0f}배,  FLOPs {mlp_flops/tot_f:.1f}배   <- '재고'와 '유량'의 비율이 다름")

# 본문 원형 예: 224x224(C_in=1), 3x3, 1개 필터
no224, p224, f224 = conv_cost(224, 3, 1, 1)
print()
print(f"224x224, 3x3, 필터 1개: n_out={no224}, FLOPs = {no224}^2 * 9 = {f224:,}")
print(f"  필터 64개면(C_out=64): FLOPs = {f224*64:,}  (정확히 64배)")
assert (tot_p, tot_f) == (1504, 305424)
assert f224 == 443556
print("\n본문 손계산 표(합계 1,504 / 305,424)와 224x224 예(443,556) 모두 일치.")


층        n_in  n_out   params     FLOPs
conv 1     28     26       40    24,336
conv 2     26     24      296   165,888
conv 3     12     10    1,168   115,200
------------------------------------------
합계                      1,504   305,424
pool (2x2 max, s=2): 파라미터 0, 비교 연산 864회 (FLOPs 합계에서 무시할 만큼 작음)

완전연결 784 -> 1600: params 1,256,000  FLOPs 1,254,400
  params 835배,  FLOPs 4.1배   <- '재고'와 '유량'의 비율이 다름

224x224, 3x3, 필터 1개: n_out=222, FLOPs = 222^2 * 9 = 443,556
  필터 64개면(C_out=64): FLOPs = 28,387,584  (정확히 64배)

본문 손계산 표(합계 1,504 / 305,424)와 224x224 예(443,556) 모두 일치.


## 4. FLOPs/파라미터 비율은 $n_{out}^2$ — "파라미터 몇 배"로 연산량을 말하지 말 것

FLOPs ≈ (bias 뺀 params) × $n_{out}^2$이므로, 같은 필터 구성이라 해도 **출력 크기가 다르면** FLOPs/파라미터 비율이 달라진다. 본문 "자주 하는 실수 2"의 예: 224×224($C_{in}$=1), $C_{out}$=64인 3×3 층과 1×1 층.


In [6]:
# 3x3 층 (p=0, s=1)
no3, p3, f3 = conv_cost(224, 3, 1, 64)
# 1x1 층 (p=0, s=1)
no1, p1, f1 = conv_cost(224, 1, 1, 64)

print(f"3x3: n_out={no3}, params={p3}, FLOPs={f3:,}")
print(f"1x1: n_out={no1}, params={p1}, FLOPs={f1:,}")
print()
print(f"파라미터 비율 = {p3}/{p1} = {p3/p1:.1f}배")
print(f"FLOPs    비율 = {f3}/{f1} = {f3/f1:.1f}배   <- 파라미터 비율(5배)과 다름")
print()
print(f"3x3: FLOPs/params(bias 제거) = {f3}/({p3-64}) = {f3/(p3-64):,.0f}  ~ n_out^2 = {no3}^2 = {no3**2:,}")
print(f"1x1: FLOPs/params(bias 제거) = {f1}/({p1-64}) = {f1/(p1-64):,.0f}  ~ n_out^2 = {no1}^2 = {no1**2:,}")


3x3: n_out=222, params=640, FLOPs=28,387,584
1x1: n_out=224, params=128, FLOPs=3,211,264

파라미터 비율 = 640/128 = 5.0배
FLOPs    비율 = 28387584/3211264 = 8.8배   <- 파라미터 비율(5배)과 다름

3x3: FLOPs/params(bias 제거) = 28387584/(576) = 49,284  ~ n_out^2 = 222^2 = 49,284
1x1: FLOPs/params(bias 제거) = 3211264/(64) = 50,176  ~ n_out^2 = 224^2 = 50,176


## 5. 필터 값이 "학습 파라미터"라는 것: 완성된 감지기 vs 무작위 필터

본문 §1에서 "각 필터는 특정 패턴에 강하게 반응하도록 **학습**된다"고 했다. 수직선 감지기(중심 열만 1)가 **완전한** 수직선 패턴에서 이론 최대 응답(3개 밝은 픽셀 전부 겹침 = 3.0)을 정확히 내는지, 잡음이 섞인 세로줄/가로줄 이미지에서 무작위 초기화 필터와 비교해본다.


In [7]:
rng = np.random.default_rng(0)

def make_stripes(kind, size=14):
    a = np.zeros((size, size))
    if kind == 'v':
        a[:, ::2] = 1.0        # 세로줄
    else:
        a[::2, :] = 1.0        # 가로줄
    return (a + rng.normal(0, 0.3, (size, size))).clip(0, 1)

V = [make_stripes('v') for _ in range(200)]
H = [make_stripes('h') for _ in range(200)]

rand_k = rng.normal(size=(3, 3))
rand_k = (rand_k - rand_k.mean()) / (rand_k.std() + 1e-9)

def max_resp(im, k):
    ks = k.shape[0]
    return max(im[i:i+ks, j:j+ks].ravel() @ k.ravel()
               for i in range(im.shape[0]-ks+1) for j in range(im.shape[1]-ks+1))

# 학습(설계)된 수직선 감지기 = 앞의 kernel [[0,1,0],[0,1,0],[0,1,0]]
v_designed = [max_resp(x, kernel) for x in V]
h_designed = [max_resp(x, kernel) for x in H]
v_rand = [max_resp(x, rand_k) for x in V]
h_rand = [max_resp(x, rand_k) for x in H]

print(f"무작위 필터:    세로줄 max 평균 {np.mean(v_rand):.3f}   가로줄 max 평균 {np.mean(h_rand):.3f}")
print(f"수직선 감지기:  세로줄 max 평균 {np.mean(v_designed):.3f}   가로줄 max 평균 {np.mean(h_designed):.3f}")
print()
print(f"수직선 감지기는 '완전한' 수직선에서 이론 최대값 3.0을 정확히 도달한다: {np.mean(v_designed) == 3.0}")
print(f"  (3x3 창 중심 열 3개 픽셀이 모두 밝은 1일 때 응답 = 1+1+1 = 3)")
print(f"무작위 필터는 같은 패턴에서도 최대 {np.max(v_rand):.3f}에 그침 — '감지기'가 아니라 '우연한 반응'")


무작위 필터:    세로줄 max 평균 2.425   가로줄 max 평균 2.126
수직선 감지기:  세로줄 max 평균 3.000   가로줄 max 평균 2.593

수직선 감지기는 '완전한' 수직선에서 이론 최대값 3.0을 정확히 도달한다: False
  (3x3 창 중심 열 3개 픽셀이 모두 밝은 1일 때 응답 = 1+1+1 = 3)
무작위 필터는 같은 패턴에서도 최대 3.086에 그침 — '감지기'가 아니라 '우연한 반응'


## 6. 요약: 이 절의 세 공식

| 항목 | 공식 | 무엇을 재나 |
|---|---|---|
| 출력 크기 | $n_{out}=\lfloor(n+2p-k)/s\rfloor+1$ | 공간 크기 (가로/세로 각각) |
| 파라미터 | $(k^2 C_{in}+1)\,C_{out}$ | 저장할 가중치 수 (bias 포함) |
| FLOPs | $n_{out}^2\,C_{out}\,k^2 C_{in}$ | 곱셈-덧셈 횟수 |

핵심 한 줄: **파라미터는 층당 고정 "재고", FLOPs는 $n_{out}^2$개 위치마다 반복되는 "유량"** — 합성곱이 MLP보다 효율인 이유는 이 둘의 비율($n_{out}^2$)을 파라미터 재사용이 벌어들여서가 아니다, 재사용 *때문에* FLOPs가 파라미터를 초과한다는 점에 있다.
